In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_dir = base_path / "dphil_paper_2/results"
out_dir = base_path / "figures"

In [ ]:
admin_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/admin_boundaries.gpkg"
admin1 = gpd.read_file(admin_boundary_path, layer="admin1")
admin1.crs
admin1.head()

In [ ]:
catchments_unionized_final = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
print("Catchments:", len(catchments))


In [ ]:
# Intersections of catchments with Admin-1 (PARISH)
intersections = gpd.overlay(
    catchments[["catchment_uid", "geometry"]],
    admin1[["PARISH", "geometry"]],
    how="intersection"
).copy()

# Areas
intersections["part_m2"] = intersections.geometry.area
catch_area_m2 = catchments.set_index("catchment_uid").geometry.area

# Summarize shares per (catchment, PARISH)
grp = (intersections
       .groupby(["catchment_uid", "PARISH"], as_index=False)
       .agg(part_m2=("part_m2", "sum")))

grp["catch_area_m2"] = grp["catchment_uid"].map(catch_area_m2)
shares = grp.assign(
    area_km2 = grp["part_m2"] / 1e6,
    share    = grp["part_m2"] / grp["catch_area_m2"],
    share_pct = lambda d: (d["share"] * 100).round(1).clip(0, 100)
).drop(columns="part_m2")

intersections

In [ ]:
# === Parish breakdowns for *every* catchment =================================

# 1) Compact per-catchment breakdown string (e.g., "ParishA (62.3%), ParishB (37.7%)")
catchment_administrative_share = shares.copy()  # use shares_nosliver if you filtered tiny slivers
catchment_administrative_share["share_pct"] = catchment_administrative_share["share_pct"].round(1)

brk_all = (catchment_administrative_share
           .sort_values(["catchment_uid", "share"], ascending=[True, False])
           .assign(entry=lambda d: d["PARISH"] + " (" + d["share_pct"].astype(str) + "%)")
           .groupby("catchment_uid", as_index=False)["entry"]
           .agg(", ".join)
           .rename(columns={"entry": "parish_breakdown"}))

# 2) Majority parish (and add area if you have it)
maj = (catchment_administrative_share.loc[catchment_administrative_share.groupby("catchment_uid")["share"].idxmax(),
                          ["catchment_uid", "PARISH", "share_pct"]]
       .rename(columns={"PARISH": "majority_parish",
                        "share_pct": "majority_share_pct"}))

areas = (catchments[["catchment_uid","area_km2"]]
         if "area_km2" in catchments.columns
         else catchments[["catchment_uid"]].assign(area_km2=catchments.geometry.area/1e6))

out = (brk_all
       .merge(maj, on="catchment_uid", how="left")
       .merge(areas, on="catchment_uid", how="left")
       .sort_values("catchment_uid"))

# 3) Save CSVs
out_csv = output_dir / "catchment_parish_breakdown_all.csv"
out.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
# Count distinct parishes per catchment
n_parishes = (shares.groupby("catchment_uid")["PARISH"]
                      .nunique()
                      .rename("n_parishes")
                      .reset_index())

# Distribution of counts
print("Parish-count distribution:\n", n_parishes["n_parishes"].value_counts().sort_index())

# Top examples with >1 parish
print("\nCatchments crossing >1 parish (top 10):")
print(n_parishes[n_parishes["n_parishes"] > 1]
      .sort_values("n_parishes", ascending=False)
      .head(10).to_string(index=False))

In [ ]:
# --- Robust top-10 catchment IDs by avoided damages --------------------------

def top_catchment_ids(n=10):
    # 1) Prefer an explicit per-catchment table named `tbl`
    if 'tbl' in globals() and 'catchment_uid' in tbl.columns:
        for col in ('avoided_ead_max', 'avoided_ead', 'avoided_ead_mid'):
            if col in tbl.columns:
                return (tbl[['catchment_uid', col]]
                        .dropna()
                        .sort_values(col, ascending=False)['catchment_uid']
                        .head(n).tolist())
    # 2) Fall back to a common table name `result`
    if 'result' in globals() and 'catchment_uid' in result.columns:
        for col in ('avoided_ead_max', 'avoided_ead', 'avoided_ead_mid'):
            if col in result.columns:
                return (result[['catchment_uid', col]]
                        .dropna()
                        .sort_values(col, ascending=False)['catchment_uid']
                        .head(n).tolist())
    # 3) Build from raw assets if available
    if 'damage_future' in globals() and {'catchment_uid','avoided_ead'}.issubset(damage_future.columns):
        agg = (damage_future.groupby('catchment_uid', as_index=False)['avoided_ead'].sum())
        return (agg.sort_values('avoided_ead', ascending=False)['catchment_uid']
                    .head(n).tolist())
    raise NameError(
        "No per-catchment avoided-EAD table found. Define `tbl` or `result` with "
        "`catchment_uid` and one of {avoided_ead_max, avoided_ead, avoided_ead_mid}, "
        "or provide `damage_future` with those columns so I can compute it."
    )

top_ids = top_catchment_ids(10)

# --- Parish % list for these top catchments ----------------------------------
brk = (shares[shares["catchment_uid"].isin(top_ids)]
       .sort_values(["catchment_uid", "share"], ascending=[True, False])
       .assign(entry=lambda d: d["PARISH"] + " (" + d["share_pct"].astype(str) + "%)")
       .groupby("catchment_uid")["entry"]
       .agg(", ".join)
       .reset_index(name="parish_breakdown"))

print(brk.to_string(index=False))

# Save the top-10 parish breakdown to CSV
out_csv = output_dir / "top10_parish_breakdown_by_catchment.csv"
brk.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
# Parish % list for the top 10 catchments in your avoided_ead_max table
top_ids = tbl["catchment_uid"].head(10).tolist()

brk = (shares[shares["catchment_uid"].isin(top_ids)]
       .sort_values(["catchment_uid", "share"], ascending=[True, False])
       .assign(entry=lambda d: d["PARISH"] + " (" + d["share_pct"].astype(str) + "%)")
       .groupby("catchment_uid")["entry"]
       .agg(", ".join)
       .reset_index(name="parish_breakdown"))

print(brk.to_string(index=False))

brk

# Save the top-10 parish breakdown to CSV
out_csv = output_dir / "top10_parish_breakdown_by_catchment.csv"
brk.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
# === Parish breakdowns for *every* catchment =================================

# 1) Compact per-catchment breakdown string (e.g., "ParishA (62.3%), ParishB (37.7%)")
shares_for_txt = shares.copy()  # use shares_nosliver if you filtered tiny slivers
shares_for_txt["share_pct"] = shares_for_txt["share_pct"].round(1)

brk_all = (shares_for_txt
           .sort_values(["catchment_uid", "share"], ascending=[True, False])
           .assign(entry=lambda d: d["PARISH"] + " (" + d["share_pct"].astype(str) + "%)")
           .groupby("catchment_uid", as_index=False)["entry"]
           .agg(", ".join)
           .rename(columns={"entry": "parish_breakdown"}))

# 2) Majority parish (and add area if you have it)
maj = (shares_for_txt.loc[shares_for_txt.groupby("catchment_uid")["share"].idxmax(),
                          ["catchment_uid", "PARISH", "share_pct"]]
       .rename(columns={"PARISH": "majority_parish",
                        "share_pct": "majority_share_pct"}))

areas = (catchments[["catchment_uid","area_km2"]]
         if "area_km2" in catchments.columns
         else catchments[["catchment_uid"]].assign(area_km2=catchments.geometry.area/1e6))

out = (brk_all
       .merge(maj, on="catchment_uid", how="left")
       .merge(areas, on="catchment_uid", how="left")
       .sort_values("catchment_uid"))

# 3) Save CSVs
out_csv = output_dir / "catchment_parish_breakdown_all.csv"
out.to_csv(out_csv, index=False)
print("Saved:", out_csv)

shares_long_csv = output_dir / "catchment_parish_shares_long_all.csv"
(shares[["catchment_uid","PARISH","area_km2","share_pct"]]
 .sort_values(["catchment_uid","share_pct"], ascending=[True, False])
 .to_csv(shares_long_csv, index=False))
print("Saved:", shares_long_csv)

wide_csv = output_dir / "catchment_parish_shares_wide_all.csv"
(shares.pivot_table(index="catchment_uid", columns="PARISH", values="share_pct", fill_value=0)
 .round(1).sort_index(axis=1).to_csv(wide_csv))
print("Saved:", wide_csv)